In [1]:
from utils import *
import squidpy as sq

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata

In [2]:
from peak_gene_utils import (
    get_top_correlated_peaks_near_gene,
    get_filtered_peak_results_by_gene
)

In [3]:
# Load cCREs as a DataFrame
ccre_bed = pd.read_csv(
    "mm10-cCREs.bed",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "ccre_id", "accession", "ccre_type"]
)

In [4]:
atac = sc.read_h5ad("desc_normalized_atac_peaks.h5ad")
expr = sc.read_h5ad("desc_normalized_rna.h5ad")

In [5]:
sp_merfish = sc.read_h5ad("../../h5ad_files/c_sp_ad.h5ad")

In [6]:
sp_merfish_names = sp_merfish.var.index.str.lower().tolist()

In [7]:
atac

AnnData object with n_obs × n_vars = 5274 × 72148
    obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density'
    var: 'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
    uns: 'files', 'gearyC', 'spatial_neighbors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'

In [8]:
data = np.load("cached_data.npz")
idx_atac = data["idx_atac"]
idx_expr = data["idx_expr"]
pdist_ = data["pdist"]

In [9]:
atac_20000 = atac[:,idx_atac]
expr_2000 = expr[:,idx_expr]

In [10]:
g_p_similarity = pdist_[20000:,:20000]

In [11]:
geary_gene_names_list = expr_2000.uns["gearyC"].head(n=300).index.tolist()

In [12]:
sq.gr.spatial_autocorr(expr_2000, mode="moran")

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  obj[key] = data


In [13]:
sq.gr.spatial_autocorr(atac_20000, mode="moran")

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  obj[key] = data


In [14]:
moran_list = expr_2000.uns["moranI"].head(n=300).index.tolist()

In [15]:
peak_moran_list = atac_20000.uns["moranI"].head(n=100).index.tolist()

In [ ]:
# Get only unannotated peaks near genes in moran_list
no_anno_results_by_gene = get_filtered_peak_results_by_gene(
    gene_list=expr_2000.var_names,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    top_n=200,
    max_distance=100000,
    keep_annotated=False
)

# Get only annotated peaks
anno_results_by_gene = get_filtered_peak_results_by_gene(
    gene_list=moran_list,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    keep_annotated=True
)

Skipping gm11128: Coordinates missing for gene gm11128.
Skipping a330050f15rik: Coordinates missing for gene a330050f15rik.
Skipping rp23-474j16.2: Coordinates missing for gene rp23-474j16.2.
Skipping rp24-136l4.6: Coordinates missing for gene rp24-136l4.6.
Skipping rp23-2d23.2: Coordinates missing for gene rp23-2d23.2.
Skipping pcnxl2: Coordinates missing for gene pcnxl2.
Skipping mir143hg: Coordinates missing for gene mir143hg.
Skipping rp24-416m6.5: Coordinates missing for gene rp24-416m6.5.
Skipping rp24-267c3.3: Coordinates missing for gene rp24-267c3.3.
Skipping rp24-134n2.1: Coordinates missing for gene rp24-134n2.1.


In [ ]:
for g in no_anno_results_by_gene.keys():
    print(g)
    print(no_anno_results_by_gene[g]["distance"])
    print()

In [ ]:
no_anno_results_by_gene[ 'slc24a2']

In [ ]:
len(anno_results_by_gene)

In [ ]:
genes_with_peaks = list(anno_results_by_gene.keys())

In [ ]:
all_peak_names = [peak for df in anno_results_by_gene.values() for peak in df.index]

In [ ]:
sq.pl.spatial_scatter(
    expr_2000, shape=None, color=genes_with_peaks, size=30
)

fam107b 3705049-3783179  3745049

In [ ]:
sq.pl.spatial_scatter(
    atac_20000, shape=None, color=all_peak_names, size=30
)

In [ ]:
sq.pl.spatial_scatter(
    atac_20000, shape=None, color=peak_moran_list, size=30
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Choose your peak
peak_name = "chr15:38907117-38907616"
#peak_name = "chr19:18972253-18972752"
peak_name = "chr1:172295659-172296158"
peak_name = "chr4:87166216-87166715"

# Find peak index
peak_idx = list(atac_20000.var_names).index(peak_name)

# Extract sparse values (cells × peaks) and convert to dense 1D array
values_sparse = atac_20000.X[:, peak_idx]
values = np.array(values_sparse.todense()).flatten()

# Get spatial coordinates
coords = atac_20000.obsm["spatial"]

# Create masks for zero and non-zero accessibility
is_zero = values < 0.2
is_nonzero = values > 0.2

# Plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(coords[is_zero, 0], coords[is_zero, 1], c="lightgray", s=5, label="zero")
sc = ax.scatter(coords[is_nonzero, 0], coords[is_nonzero, 1], c=values[is_nonzero],
                cmap="viridis", s=5)
plt.colorbar(sc, ax=ax).set_label("Peak Accessibility")
ax.set_title(peak_name)
ax.set_xlabel("spatial1")
ax.set_ylabel("spatial2")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
peak_to_index = {name: i for i, name in enumerate(atac_20000.var_names)}

In [ ]:
sv_peak_indices = [peak_to_index[p] for p in peak_moran_list if p in peak_to_index]

In [ ]:
atac_20000.var.iloc[19984]

In [ ]:
# Transpose matrix so we go peak → gene
similarity_T = g_p_similarity.T  # shape: [peaks × genes]

# Store results in a dict
top_genes_by_peak = {}

for peak_idx in sv_peak_indices:
    correlations = similarity_T[peak_idx]  # vector of gene similarities
    top_gene_indices = np.argsort(correlations)[::-1][:10]  # top 10
    top_gene_names = expr_2000.var_names[top_gene_indices].tolist()
    top_genes_by_peak[atac_20000.var_names[peak_idx]] = top_gene_names

In [ ]:
top_genes_by_peak = {}

for peak_idx in sv_peak_indices:
    peak_name = atac_20000.var_names[peak_idx]
    peak_row = atac_20000.var.loc[peak_name]
    
    peak_chr = peak_row["chrom"]
    peak_mid = (peak_row["start"] + peak_row["end"]) // 2

    correlations = similarity_T[peak_idx]
    top_gene_indices = np.argsort(correlations)[::-1]

    filtered_genes = []

    for gene_idx in top_gene_indices:
        gene_name = expr_2000.var_names[gene_idx]
        gene_row = expr_2000.var.loc[gene_name]

        if gene_row["chrom"] != peak_chr:
            continue  # wrong chromosome

        gene_mid = (gene_row["start"] + gene_row["end"]) // 2
        distance = abs(peak_mid - gene_mid)

        if distance <= 100_000:  # distance threshold in base pairs
            filtered_genes.append(gene_name)

        if len(filtered_genes) == 10:
            break  # stop once we have 10 matching genes

    top_genes_by_peak[peak_name] = filtered_genes


In [ ]:
genes_associated_with_SV_peaks = list(set([gene for genes in top_genes_by_peak.values() for gene in genes]))

In [ ]:
sq.pl.spatial_scatter(
    expr_2000, shape=None, color=genes_associated_with_SV_peaks, size=30
)

In [ ]:
top_genes_by_peak

chr18:67650853-67651352 cep76

In [ ]:
sq.pl.spatial_scatter(atac_20000, shape=None, color="chr18:67650853-67651352")

In [ ]:
genes_associated_with_SV_peaks

In [ ]:
sp_merfish_names

In [ ]:
sc.tl.pca(atac_20000)

In [ ]:
sc.pl.pca_variance_ratio(atac_20000, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    atac_20000, color="subclass"
)

In [ ]:
# nearest neighbor graph
sc.pp.neighbors(atac_20000, n_pcs=50)
nn_graph_genes = atac_20000.obsp["connectivities"]
# spatial proximity graph
sq.gr.spatial_neighbors(atac_20000)
nn_graph_space = atac_20000.obsp["spatial_connectivities"]

In [ ]:
alpha = 0.2

joint_graph = (1 - alpha) * nn_graph_genes + alpha * nn_graph_space
sc.tl.leiden(atac_20000, adjacency=joint_graph, key_added="squidpy_domains")

In [ ]:
atac_20000

In [ ]:
sq.pl.spatial_scatter(atac_20000, shape=None, color=["subclass", "squidpy_domains"], size=8)

In [ ]:
sc.pl.pca(
    atac_20000, color="squidpy_domains"
)

In [ ]:
# Run Leiden clustering to identify spatial domains
sc.tl.leiden(atac_20000, resolution=0.5)

In [ ]:
# UMAP visualization of the nichePCA embedding
sc.tl.umap(atac_20000)
sc.pl.umap(atac_20000, color="leiden")

In [ ]:
sq.pl.spatial_scatter(atac_20000, shape=None, color=["leiden", "squidpy_domains"], size=8)

In [ ]:
sc.pl.umap(atac_20000, color=["squidpy_domains", "leiden"])